# Assumption checking short use cases

## Pre-modeling assumptions

In [1]:
import pandas as pd

dataset_path = "../data/all_data/abortion_bf15.csv"
df = pd.read_csv(dataset_path)
df.head()

,fip,age,race,year,sex,totcase,totpop,rate,totrate,id,...,female,lnr,t,younger,fa,pi,wm15,wf15,bm15,bf15
0,1.0,15.0,2.0,1985.0,2,5683.0,106187,6527.5,5351.9,14.0,...,1.0,8.783779,1.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0
1,1.0,15.0,2.0,1986.0,2,5344.0,106831,6351.2,5002.3,14.0,...,1.0,8.756399,2.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0
2,1.0,15.0,2.0,1987.0,2,4983.0,106496,5759.1,4679.0,14.0,...,1.0,8.658537,3.0,1.0,1.0,1.0,0.0,0.0,0.0,1.0
3,1.0,15.0,2.0,1988.0,2,5276.0,105238,6139.6,5013.4,14.0,...,1.0,8.722515,4.0,1.0,1.0,1.0,0.0,0.0,0.0,1.0
4,1.0,15.0,2.0,1989.0,2,5692.0,102956,5951.5,5528.6,14.0,...,1.0,8.691399,5.0,1.0,1.0,1.0,0.0,0.0,0.0,1.0


#### Cond-ignorability (IPW, matching)

In [2]:
from cais.methods.pre_model_assumption_utils import check_cond_ignorability

C:\Users\beriv\PycharmProjects\causal-agent\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
covariates = ['crack', 'alcohol', 'income', 'ur', 'poverty', 'black', 'perc1519']
result_cond_ignorability = check_cond_ignorability(df, 'repeal', covariates)
print(result_cond_ignorability)

{'passed': False, 'reasoning': "Randomization check on 7 covariates (|SMD| < 0.1). Imbalanced: ['crack', 'alcohol', 'income', 'ur', 'poverty', 'black', 'perc1519'].", 'details': {'smds': {'crack': np.float64(0.1901831312076417), 'alcohol': np.float64(0.3090489010094973), 'income': np.float64(0.5807936072744352), 'ur': np.float64(0.4436246174561076), 'poverty': np.float64(-0.2878371568806569), 'black': np.float64(-0.630179818114365), 'perc1519': np.float64(-0.5894732981847037)}, 'threshold': 0.1, 'imbalanced': {'crack': np.float64(0.1901831312076417), 'alcohol': np.float64(0.3090489010094973), 'income': np.float64(0.5807936072744352), 'ur': np.float64(0.4436246174561076), 'poverty': np.float64(-0.2878371568806569), 'black': np.float64(-0.630179818114365), 'perc1519': np.float64(-0.5894732981847037)}}}


In [4]:
from cais.methods.pre_model_assumption_utils import check_positivity
from cais.methods.propensity_score.base import estimate_propensity_scores

In [5]:
ps = estimate_propensity_scores(df, 'repeal', covariates)

overlap_result = check_positivity(df, 'repeal', ps)
print(overlap_result)

{'passed': False, 'reasoning': 'Overlap proportion: 0.777 (threshold 0.5). 472 obs (64.0%) outside [0.1, 0.9]. Consider trimming or restricting to common support.', 'details': {'treated_range': (0.01873818828478758, 0.766475780981912), 'control_range': (0.01, 0.6066358199069207), 'overlap_range': (0.01873818828478758, 0.6066358199069207), 'overlap_proportion': 0.7771532762873611, 'sufficient_overlap': np.True_, 'n_extreme_ps': 472, 'pct_extreme_ps': 0.6404341926729986}}


### Fix an OPENAI_API_KEY in environment before running cells below

In [2]:
from cais.config import get_llm_client
llm = get_llm_client()

In [7]:
from cais.methods.pre_model_assumption_utils import check_permissive_sutva

dataset_description = (
    "Panel data of 51 US states from 1985-2000 (Donohue & Levitt). "
    "Treatment 'repeal' is binary: whether the state legalized abortion "
    "before Roe v. Wade in 1973. Outcome 'rate' is the crime rate per 100k. "
    "The hypothesis is that legalized abortion reduced unwanted births, "
    "which later reduced crime. Subgroup: black females age 15."
)

variables_info = {
    'treatment': 'repeal',
    'outcome': 'rate',
    'covariates': covariates,
    'panel_id': 'fip (state FIPS code)',
    'time': 'year (1985-2000)',
}

result_sutva = check_permissive_sutva(dataset_description, variables_info, llm=llm)
print(result_sutva)

{'passed': True, 'reasoning': "The assumption of SUTVA is plausibly satisfied in this context as the treatment 'repeal' pertains to state-level legalization of abortion, which is unlikely to directly affect the potential outcomes of other states. Additionally, the treatment appears to be uniformly applied within states that legalized abortion, minimizing concerns about hidden versions of the treatment. The analysis focuses on a specific subgroup (black females age 15), which further supports the consistency of treatment application within that group.", 'details': {'assumption': 'SUTVA (Stable Unit Treatment Value Assumption)'}}


#### Instrumental Variables (IVs)

In [8]:
from cais.methods.pre_model_assumption_utils import (
    check_iv_relevance,
    check_iv_exclusion,
    check_iv_exogeneity,
    check_iv_monotonicity,
)

In [9]:
dataset_iv_path = "../data/all_data/card_geographic.csv"
df_iv = pd.read_csv(dataset_iv_path)

In [10]:
df_iv = df_iv.drop(columns=['Unnamed: 0'], errors='ignore')
df_iv = df_iv.dropna()

In [11]:
df_iv.head()

,nearc4,educ,black,smsa,south,married,exper,lwage
0,0,7,1,1,0,1.0,16,6.306275
1,0,12,0,1,0,1.0,9,6.175867
2,0,12,0,1,0,1.0,16,6.580639
3,1,11,0,1,0,1.0,10,5.521461
4,1,12,0,1,0,1.0,16,6.591674


In [12]:
# --- check_iv_relevance : Is nearc4 a strong instrument for educ ? ---
result_iv_relevance = check_iv_relevance(
    df=df_iv,
    treatment='educ',
    instruments=['nearc4'],
    covariates=['black', 'smsa', 'south', 'married', 'exper'],
)
print(result_iv_relevance)

{'passed': True, 'reasoning': 'First-stage F = 15.77 (threshold 10.0). Strong instrument.', 'details': {'f_statistic': 15.766661138654587, 'p_value': 7.333887270678314e-05, 'threshold': 10.0}}


In [13]:
# --- check_iv_exclusion ---
card_description = (
    "Card (1995) dataset. 3010 men from the NLS Young Men Cohort. "
    "Treatment is 'educ' (years of education). Instrument is 'nearc4' "
    "(grew up near a 4-year college). Outcome is 'lwage' (log wage). "
    "The exclusion restriction argument is that college proximity affects "
    "education but not wages directly — though this is debated, since "
    "proximity may correlate with local labor market conditions."
)

card_variables = {
    'treatment': 'educ',
    'outcome': 'lwage',
    'instrument': 'nearc4',
    'covariates': ['black', 'smsa', 'south', 'married', 'exper'],
}

result_iv_exclusion = check_iv_exclusion(card_description, card_variables, llm=llm)
print("check_iv_exclusion :", result_iv_exclusion)

check_iv_exclusion : {'passed': False, 'reasoning': 'The exclusion restriction is likely violated because growing up near a 4-year college may influence local labor market conditions, which could directly affect wages (lwage) independent of education (educ). This suggests that proximity to a college could have a direct effect on wages, thus undermining the validity of the instrument.', 'details': {'assumption': 'Exclusion restriction'}}


In [14]:
# --- check_iv_exogeneity ---
result_iv_exogeneity = check_iv_exogeneity(card_description, card_variables, llm=llm)
print("check_iv_exogeneity :", result_iv_exogeneity)

check_iv_exogeneity : {'passed': False, 'reasoning': "The assumption of instrument exogeneity is likely violated in this context because 'nearc4' (growing up near a 4-year college) may be correlated with local labor market conditions, which can affect wages independently of education. This suggests that 'nearc4' could influence 'lwage' through pathways other than education, undermining the validity of the exclusion restriction.", 'details': {'assumption': 'Instrument exogeneity (independence)'}}


In [15]:
# --- check_iv_monotonicity ---
result_iv_monotonicity = check_iv_monotonicity(card_description, card_variables, llm=llm)
print("check_iv_monotonicity :", result_iv_monotonicity)

check_iv_monotonicity : {'passed': False, 'reasoning': 'The assumption of monotonicity is likely violated in this context because proximity to a 4-year college may lead some individuals to pursue more education while others may be discouraged or choose not to attend college despite proximity. This suggests the presence of defiers, as some individuals could be negatively influenced by the local educational environment, leading to a decrease in education despite being near a college.', 'details': {'assumption': 'Monotonicity (LATE)'}}


In the description furnished before, there wasn't enough description for the LLM to judge whether IV monocity was valid. That's why it returns `'None'`.

We enrich the description now:

In [16]:
card_description = (
    "Card (1995) dataset. 3010 men from the NLS Young Men Cohort. "
    "Treatment is 'educ' (years of education). Instrument is 'nearc4' "
    "(grew up near a 4-year college). Outcome is 'lwage' (log wage). "
    "The instrument works through reduced cost of attending college: "
    "individuals near a college face lower transportation and housing costs, "
    "making them more likely to attend. It is implausible that proximity "
    "to a college would cause someone to get LESS education — the effect "
    "should go in one direction only (more proximity → more education or no change)."
)

In [17]:
# --- check_iv_monotonicity ---
result_iv_monotonicity = check_iv_monotonicity(card_description, card_variables, llm=llm)
print("check_iv_monotonicity :", result_iv_monotonicity)

check_iv_monotonicity : {'passed': True, 'reasoning': 'The assumption of monotonicity is plausibly satisfied in this context because proximity to a 4-year college is expected to either increase education or have no effect, but not decrease it. Given that the mechanism involves reduced costs associated with attending college, it is reasonable to conclude that no individuals would be deflected from pursuing education due to their proximity to a college.', 'details': {'assumption': 'Monotonicity (LATE)'}}


#### Difference-in-Differences (DiD)

In [18]:
from cais.methods.pre_model_assumption_utils import (
    check_parallel_trends,
    check_no_anticipation,
    check_baseline_outcome_balance,
    check_stable_group_composition,
)

In [19]:
df_did = pd.read_csv('../data/all_data/castle.csv')
df_did['ever_treated'] = df_did.groupby('sid')['post'].transform('max').astype(int)

# 2006 is the cutoff year for the tests
treatment_year = 2006

df_did.head()

,state,year,sid,cdl,pre2_cdl,caselaw,anywhere,assumption,civil,homicide_c,...,_Iyear_2003,_Iyear_2004,_Iyear_2005,_Iyear_2006,_Iyear_2007,_Iyear_2008,_Iyear_2009,_Iyear_2010,popwt,ever_treated
0,Alabama,2000,1,0.0,0.0,0.0,0,0,0,329,...,0,0,0,0,0,0,0,0,4499293.0,1
1,Alabama,2001,1,0.0,0.0,0.0,0,0,0,379,...,0,0,0,0,0,0,0,0,4499293.0,1
2,Alabama,2002,1,0.0,0.0,0.0,0,0,0,303,...,0,0,0,0,0,0,0,0,4499293.0,1
3,Alabama,2003,1,0.0,0.0,0.0,0,0,0,299,...,1,0,0,0,0,0,0,0,4499293.0,1
4,Alabama,2004,1,0.0,1.0,0.0,0,0,0,254,...,0,1,0,0,0,0,0,0,4499293.0,1


In [20]:
# --- check_parallel_trends ---
result_parallel_trends = check_parallel_trends(
    df=df_did,
    time_var='year',
    outcome='l_homicide',
    group_indicator_col='ever_treated',
    treatment_period_start=treatment_year,
)
print(result_parallel_trends)

{'passed': np.True_, 'reasoning': 'Simple linear trend test: p-value for group-trend interaction: 0.8264. Parallel trends: True.', 'details': {'p_value': np.float64(0.8263558702861501), 'error': None}}


In [21]:
# --- check_no_anticipation ---
covariates = ['l_police', 'l_income', 'l_prisoner', 'unemployrt', 'poverty']

result_no_anticipation = check_no_anticipation(
    df=df_did,
    time_var='year',
    group_var='sid',
    outcome='l_homicide',
    treated_unit_indicator='ever_treated',
    covariates=covariates,
    treatment_period_start=treatment_year,
    placebo_period_start=2003,
)
print(result_no_anticipation)

{'passed': True, 'reasoning': 'Placebo treatment effect estimated at 0.0976 (p=0.1673). Test passed: True.', 'details': {'passed': True, 'effect_estimate': 0.09764229987805968, 'p_value': 0.16731301316128822, 'error': None}}


In [22]:
# --- check_baseline_outcome_balance ---
result_baseline_outcome_balance = check_baseline_outcome_balance(
    df=df_did,
    treatment='ever_treated',
    outcome='l_homicide',
    time_var='year',
    treatment_period_start=treatment_year,
)
print(result_baseline_outcome_balance)

{'passed': np.False_, 'reasoning': 'Baseline outcome SMD = 0.675 (threshold 0.1).', 'details': {'smd_pre_outcome': np.float64(0.6753481392211624), 'threshold': 0.1}}


In [23]:
# --- check_stable_group_composition (LLM-reasoned) ---
castle_description = (
    "Castle Doctrine dataset. Panel of 50 US states from 2000-2010. "
    "Treatment is adoption of Castle Doctrine / Stand Your Ground laws, "
    "which expanded the right to use lethal force in self-defense. "
    "States adopted the law at different times (staggered treatment). "
    "Outcome is log homicide rate. Unit of observation is state-year."
)

castle_variables = {
    'treatment': 'ever_treated',
    'outcome': 'l_homicide',
    'panel_id': 'sid',
    'time': 'year (2000-2010)',
    'covariates': covariates,
}

result_stable_group_composition = check_stable_group_composition(castle_description, castle_variables, llm=llm)
print(result_stable_group_composition)

{'passed': False, 'reasoning': 'The assumption of stable group composition is likely violated in this analysis due to the staggered adoption of the Castle Doctrine laws across states. This staggered treatment could lead to differential attrition or selective entry/exit, as states that adopt the laws may differ systematically from those that do not, potentially affecting the composition of treatment and control groups over time.', 'details': {'assumption': 'Stable group composition'}}


In [24]:
castle_description_real = (
    "Castle Doctrine dataset. Panel of 50 US states from 2000-2010. "
    "Treatment is adoption of Castle Doctrine / Stand Your Ground laws. "
    "Outcome is log homicide rate. Unit of observation is state-year. "
    "All 50 states are observed for all 11 years — the panel is balanced "
    "with no missing state-year observations. States do not enter or exit "
    "the dataset. However, population migration between states could change "
    "the demographic composition of treatment and control states over time."
)

result_stable_group_composition2 = check_stable_group_composition(castle_description, castle_variables, llm=llm)
print(result_stable_group_composition2)

{'passed': False, 'reasoning': 'The assumption of stable group composition is likely violated in this analysis due to the staggered adoption of the Castle Doctrine laws across states. This staggered treatment could lead to differential attrition or selective entry/exit of states based on their homicide rates or other unobserved factors, which may change the composition of treatment and control groups over time.', 'details': {'assumption': 'Stable group composition'}}


In [25]:
# --- check_sutva (LLM-reasoned) ---

castle_description_real = (
    "Castle Doctrine dataset. Panel of 50 US states from 2000-2010. "
)

result_sutva_did = check_permissive_sutva(castle_description, castle_variables, llm=llm)
print(result_sutva_did)

{'passed': True, 'reasoning': 'The Castle Doctrine laws are state-level policies that do not directly affect other states, suggesting that interference between units is minimal. Additionally, the treatment (adoption of the law) is clearly defined and applied consistently across states, supporting the assumption of no hidden versions of the treatment.', 'details': {'assumption': 'SUTVA (Stable Unit Treatment Value Assumption)'}}


#### Frontdoor adjustment

In [1]:
from cais.methods.pre_model_assumption_utils import (
    check_frontdoor_full_mediation,
    check_frontdoor_no_TM_confounding,
    check_frontdoor_T_blocks_MY,
    check_frontdoor_positivity,
)

C:\Users\beriv\PycharmProjects\causal-agent\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Description classique : smoking → tar → cancer
frontdoor_description = (
    "Observational study examining the effect of smoking (T) on lung cancer (Y). "
    "Tar deposits in lungs (M) are used as a mediator. The argument is that "
    "smoking affects cancer only through tar accumulation. There may be "
    "unobserved genetic confounders affecting both smoking behavior and cancer risk."
)

frontdoor_variables = {
    'treatment': 'smoking',
    'mediator': 'tar deposits',
    'outcome': 'lung cancer',
    'potential_confounders': 'genetic predisposition (unobserved)',
}

In [4]:
# --- Full mediation : T→Y only through M ? ---
result = check_frontdoor_full_mediation(frontdoor_description, frontdoor_variables, llm=llm)
print("check_frontdoor_full_mediation :", result)

check_frontdoor_full_mediation : {'passed': False, 'reasoning': 'The assumption of full mediation is not plausibly satisfied because there are unobserved genetic confounders that may influence both smoking behavior and lung cancer risk. This suggests that there could be a direct effect of smoking on lung cancer that is not fully captured by the mediator of tar deposits, violating the full mediation assumption.', 'details': {'assumption': 'Full mediation'}}


In [5]:
# --- No T-M confounding ---
result = check_frontdoor_no_TM_confounding(frontdoor_description, frontdoor_variables, llm=llm)
print("check_frontdoor_no_TM_confounding :", result)

check_frontdoor_no_TM_confounding : {'passed': False, 'reasoning': 'The assumption of no T-M confounding is not plausibly satisfied because there are unobserved genetic predispositions that could influence both smoking behavior and the accumulation of tar deposits in the lungs. These genetic factors could create a confounding relationship between the treatment (smoking) and the mediator (tar deposits), thus violating the assumption.', 'details': {'assumption': 'No T-M confounding'}}


In [6]:
# --- T blocks M→Y confounding ---
result = check_frontdoor_T_blocks_MY(frontdoor_description, frontdoor_variables, llm=llm)
print("check_frontdoor_T_blocks_MY :", result)

check_frontdoor_T_blocks_MY : {'passed': False, 'reasoning': 'The assumption that T blocks M→Y confounding is not plausibly satisfied because there are unobserved genetic confounders that affect both smoking behavior (T) and lung cancer risk (Y). These unobserved confounders can create back-door paths between the mediator (M) and the outcome (Y), violating the assumption.', 'details': {'assumption': 'T blocks M→Y confounding'}}


In [8]:
from cais.methods.pre_model_assumption_utils import check_frontdoor_positivity
import numpy as np
import pandas as pd

# Simulate frontdoor data
np.random.seed(42)
n = 500
df_fd = pd.DataFrame({
    'T': np.random.binomial(1, 0.5, n),
    'M': np.random.binomial(1, 0.6, n),
})

# Verified
result = check_frontdoor_positivity(df_fd, 'T', 'M')
print("Normal case :", result)

Normal case : {'passed': True, 'reasoning': '4/4 (treatment, mediator) combinations observed. 0 empty, 0 sparse (< 5 obs). Positivity satisfied.', 'details': {'total_combos': 4, 'observed_combos': 4, 'empty': 0, 'sparse': 0, 'min_count': 5}}


In [9]:
# Violation: an empty combination
df_fd_bad = df_fd.copy()
df_fd_bad = df_fd_bad[~((df_fd_bad['T'] == 1) & (df_fd_bad['M'] == 0))]

result = check_frontdoor_positivity(df_fd_bad, 'T', 'M')
print("Violation case :", result)

Violation case : {'passed': False, 'reasoning': '3/4 (treatment, mediator) combinations observed. 1 empty, 0 sparse (< 5 obs). Some combinations are empty or near-empty — frontdoor formula may be undefined.', 'details': {'total_combos': 4, 'observed_combos': 3, 'empty': 1, 'sparse': 0, 'min_count': 5}}


## Post-modeling assumptions

In [31]:
from cais.methods.post_model_assumption_utils import (
    check_iv_overidentification, check_balance_after_matching, check_balance_after_weighting
)

#### Balance checks (IPW, matching)

Let's perform matching on the `abortion_bf15.csv` dataset

In [45]:
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import NearestNeighbors
import numpy as np

covariates = ['crack', 'alcohol', 'income', 'ur', 'poverty', 'black', 'perc1519']
df_clean = df[covariates + ['repeal', 'rate']].dropna()
T = df_clean['repeal'].astype(int)

# Estimate propensity scores
ps_model = LogisticRegression(max_iter=1000)
ps_model.fit(df_clean[covariates], T)
ps = ps_model.predict_proba(df_clean[covariates])[:, 1]
ps = np.clip(ps, 0.01, 0.99)

print(f"Observations : {len(df_clean)}")
print(f"Traited : {T.sum()}, Controls : {(1-T).sum()}")

treated_idx = df_clean[T == 1].index
control_idx = df_clean[T == 0].index

nn = NearestNeighbors(n_neighbors=1)
nn.fit(ps[T == 0].reshape(-1, 1))
distances, indices = nn.kneighbors(ps[T == 1].reshape(-1, 1))

matched_control_idx = control_idx[indices.flatten()]
df_matched = pd.concat([df_clean.loc[treated_idx], df_clean.loc[matched_control_idx]])

Observations : 737
Traited : 80, Controls : 657


In [41]:
# --- check_balance_after_matching ---
result = check_balance_after_matching(df_matched, 'repeal', covariates)
print("check_balance_after_matching :", result)

check_balance_after_matching : {'passed': False, 'reasoning': "Matched sample balance on 7 covariates. Still imbalanced: ['crack', 'poverty', 'perc1519'].", 'details': {'smds': {'crack': np.float64(-0.35194822827546235), 'alcohol': np.float64(0.09820595193655905), 'income': np.float64(-0.08401277897298327), 'ur': np.float64(0.0653372997131283), 'poverty': np.float64(0.20211127320121058), 'black': np.float64(-0.0014753254554432944), 'perc1519': np.float64(0.20136611611785302)}, 'threshold': 0.1, 'imbalanced': {'crack': np.float64(-0.35194822827546235), 'poverty': np.float64(0.20211127320121058), 'perc1519': np.float64(0.20136611611785302)}}}


Let's perform matching on the `lalonde_data_psid.csv` dataset

In [46]:
df_lalonde = pd.read_csv('../data/all_data/lalonde_data_psid.csv').dropna()
covariates_lal = ['age', 'education', 'black', 'hispanic', 'married', 'nodegree', 're74', 're75']
T_lal = df_lalonde['treat'].astype(int)

ps_lal = LogisticRegression(max_iter=1000).fit(df_lalonde[covariates_lal], T_lal).predict_proba(df_lalonde[covariates_lal])[:, 1]
ps_lal = np.clip(ps_lal, 0.01, 0.99)

weights_lal = np.where(T_lal == 1, 1 / ps_lal, 1 / (1 - ps_lal))

result = check_balance_after_weighting(df_lalonde, 'treat', covariates_lal, weights_lal)
print("IPW balance :", result)

IPW balance : {'passed': False, 'reasoning': "Weighted balance on 8 covariates. Still imbalanced: ['age'].", 'details': {'weighted_smds': {'age': np.float64(-0.14624458496355314), 'education': np.float64(0.015037166411965392), 'black': np.float64(0.08334646157682368), 'hispanic': np.float64(-0.03105377837487486), 'married': np.float64(-0.07682990482566496), 'nodegree': np.float64(0.015280318253117237), 're74': np.float64(-0.06313795380968303), 're75': np.float64(-0.004874393257471577)}, 'threshold': 0.1, 'imbalanced': {'age': np.float64(-0.14624458496355314)}}}


Let's perform IPW on the `abortion_bf15.csv` dataset

In [43]:
from cais.methods.propensity_score.base import estimate_propensity_scores

# Estimate PS
ps = estimate_propensity_scores(df_clean, 'repeal', covariates)
ps = np.clip(ps, 0.01, 0.99)
T = df_clean['repeal'].astype(int)

# Computing IPW weights
weights = np.where(T == 1, 1 / ps, 1 / (1 - ps))

In [44]:
# --- check_balance_after_weighting (IPW) ---
result = check_balance_after_weighting(df_clean, 'repeal', covariates, weights)
print(result)

{'passed': False, 'reasoning': "Weighted balance on 7 covariates. Still imbalanced: ['crack', 'alcohol', 'ur', 'poverty', 'black', 'perc1519'].", 'details': {'weighted_smds': {'crack': np.float64(0.34447889210407), 'alcohol': np.float64(0.40374532829144905), 'income': np.float64(0.010660875506205408), 'ur': np.float64(-0.23481605164645944), 'poverty': np.float64(-0.26256027859824893), 'black': np.float64(-0.3988844819819745), 'perc1519': np.float64(-0.3584176988065902)}, 'threshold': 0.1, 'imbalanced': {'crack': np.float64(0.34447889210407), 'alcohol': np.float64(0.40374532829144905), 'ur': np.float64(-0.23481605164645944), 'poverty': np.float64(-0.26256027859824893), 'black': np.float64(-0.3988844819819745), 'perc1519': np.float64(-0.3584176988065902)}}}


Let's perform IPW on the `lalonde_data_psid.csv` dataset

In [47]:
nn = NearestNeighbors(n_neighbors=1)
nn.fit(ps_lal[T_lal == 0].reshape(-1, 1))
distances, indices = nn.kneighbors(ps_lal[T_lal == 1].reshape(-1, 1))

treated_idx = df_lalonde[T_lal == 1].index
control_idx = df_lalonde[T_lal == 0].index
matched_control_idx = control_idx[indices.flatten()]
df_matched_lal = pd.concat([df_lalonde.loc[treated_idx], df_lalonde.loc[matched_control_idx]])

result = check_balance_after_matching(df_matched_lal, 'treat', covariates_lal)
print("Matching balance (LaLonde) :", result)

Matching balance (LaLonde) : {'passed': False, 'reasoning': "Matched sample balance on 8 covariates. Still imbalanced: ['age'].", 'details': {'smds': {'age': np.float64(0.10258472633527428), 'education': np.float64(0.00504611249078997), 'black': np.float64(0.09928459218720592), 'hispanic': np.float64(-0.08443761220875926), 'married': np.float64(0.0), 'nodegree': np.float64(0.02359013327218533), 're74': np.float64(0.043937949151885126), 're75': np.float64(0.05707051827285377)}, 'threshold': 0.1, 'imbalanced': {'age': np.float64(0.10258472633527428)}}}


#### Instrumental Variables (IVs)

In [34]:
result_iv_overidentification = check_iv_overidentification(
    sm_results=None,
    df=df_iv,
    treatment='educ',
    outcome='lwage',
    instruments=['nearc4'],
    covariates=['black', 'smsa', 'south', 'married', 'exper'],
)
print(result_iv_overidentification)

{'passed': None, 'reasoning': 'Test not applicable (Need more instruments than endogenous regressors)', 'details': {}}


In [35]:
from statsmodels.sandbox.regression.gmm import IV2SLS
import statsmodels.api as sm

# using 'nearc4' and 'smsa' as instruments (smsa isn't a real instrument, just used for testing)
instruments = ['nearc4', 'smsa']
covariates_iv = ['black', 'south', 'married', 'exper']

# IV estimation via 2SLS
endog = df_iv['lwage']
exog = sm.add_constant(df_iv[['educ'] + covariates_iv])
instrument_matrix = sm.add_constant(df_iv[instruments + covariates_iv])

iv_model = IV2SLS(endog, exog, instrument_matrix).fit()

result_iv_overidentification2 = check_iv_overidentification(
    sm_results=iv_model,
    df=df_iv,
    treatment='educ',
    outcome='lwage',
    instruments=instruments,
    covariates=covariates_iv,
)
print(result_iv_overidentification2)

{'passed': np.False_, 'reasoning': 'Sargan-Hansen test: statistic=8.72, p=0.0031. Instruments may be invalid — correlated with errors.', 'details': {'statistic': np.float64(8.724238008268228), 'p_value': np.float64(0.0031400727800432876), 'status': 'Test successful'}}
